In [6]:
!git clone https://github.com/chenyez/ZeroStance.git
%cd ZeroStance
!pip install transformers sentencepiece tweet-preprocessor wordninja tensorboard scikit-learn pandas tqdm -q

Cloning into 'ZeroStance'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 229 (delta 4), reused 0 (delta 0), pack-reused 218 (from 2)
Receiving objects: 100% (229/229), 61.98 MiB | 23.12 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Updating files: 100% (212/212), done.
/kaggle/working/ZeroStance
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 9.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [7]:
import os
import pandas as pd

df = pd.read_csv("data/covid19/raw_train_all_onecol.csv", encoding='ISO-8859-1')
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("Label distribution:\n", df['Stance 1'].value_counts())
print(df.head(3))

Columns: ['Tweet', 'Target 1', 'Stance 1', 'Opinion Towards', 'Sentiment', 'Tweet Id']
Shape: (4533, 6)
Label distribution:
 Stance 1
NONE       1627
FAVOR      1464
AGAINST    1442
Name: count, dtype: int64
                                               Tweet    Target 1 Stance 1  \
0  @SuzanneEvans1 @Jo_Turandot @LadyTrumpington S...  face masks    FAVOR   
1  I try to be unbiased on my Twitter.\n\nWhich i...  face masks    FAVOR   
2  @GOPLeader So 9 out of 10 members of Congress ...  face masks    FAVOR   

                                     Opinion Towards Sentiment      Tweet Id  
0  1.  The tweet explicitly expresses opinion abo...       pos  1.280000e+18  
1  1.  The tweet explicitly expresses opinion abo...       pos  1.280000e+18  
2  2. The tweet does NOT expresses opinion about ...       neg  1.290000e+18  


In [10]:
import os
print(os.listdir("/kaggle/working/ZeroStance/data/covid19"))

['raw_train_all_onecol.csv', 'raw_train_all_onecol_bt.csv', 'raw_test_all_onecol.csv', 'raw_train_all_onecol_eda.csv', 'raw_val_all_onecol.csv', 'raw_train_all_onecol_asda.csv']


In [12]:
# Fix the import in train_model_v2.py
with open("/kaggle/working/ZeroStance/src/train_model_v2.py", "r") as f:
    content = f.read()

# Replace transformers AdamW with torch's AdamW
content = content.replace(
    "from transformers import AdamW",
    "from torch.optim import AdamW"
)

with open("/kaggle/working/ZeroStance/src/train_model_v2.py", "w") as f:
    f.write(content)

print("Fixed! Verifying...")
# Confirm the fix
!grep -n "AdamW" /kaggle/working/ZeroStance/src/train_model_v2.py

Fixed! Verifying...
13:from torch.optim import AdamW


In [22]:
with open("/kaggle/working/ZeroStance/src/utils/data_helper.py", "r") as f:
    content = f.read()

# Fix 1: Replace batch_encode_plus with __call__ (modern equivalent)
content = content.replace(
    "tokenizer.batch_encode_plus(",
    "tokenizer("
)

# Fix 2: Replace the RoBERTa local model path with bertweet-base
content = content.replace(
    "AutoTokenizer.from_pretrained('vinai/bertweet-base')",
    "AutoTokenizer.from_pretrained('vinai/bertweet-base')"
)

with open("/kaggle/working/ZeroStance/src/utils/data_helper.py", "w") as f:
    f.write(content)

print("Fixed! Verifying...")
!grep -n "batch_encode_plus\|from_pretrained" /kaggle/working/ZeroStance/src/utils/data_helper.py

Fixed! Verifying...
38:        tokenizer = BertweetTokenizer.from_pretrained("vinai/bertweet-base", normalization=True)
40:        tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-mnli", local_files_only=True)
42:#         tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
43:#         tokenizer = BertTokenizer.from_pretrained("../../model_hub/covid-twitter-bert-v2")
44:        tokenizer = AutoTokenizer.from_pretrained("../../model_hub/twhin-bert-large")
46:        tokenizer = AutoTokenizer.from_pretrained('vinai/bertweet-base')
47:        # tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-large", local_files_only=True)
49:        tokenizer = RobertaTokenizer.from_pretrained('roberta-large-mnli')


In [23]:
with open("/kaggle/working/ZeroStance/src/utils/model_utils.py", "r") as f:
    content = f.read()

content = content.replace(
    "../../model_hub/bertweet-large",
    "vinai/bertweet-base"
)
content = content.replace(
    "../../model_hub/twhin-bert-large",
    "vinai/bertweet-base"
)

with open("/kaggle/working/ZeroStance/src/utils/model_utils.py", "w") as f:
    f.write(content)

print("Fixed! Verifying...")
!grep -n "from_pretrained" /kaggle/working/ZeroStance/src/utils/model_utils.py

Fixed! Verifying...


In [25]:
with open("/kaggle/working/ZeroStance/src/utils/modeling.py", "r") as f:
    content = f.read()

# Fix all local model_hub paths to use HuggingFace hub
content = content.replace(
    "AutoConfig.from_pretrained('../../model_hub/bertweet-large', local_files_only=True)",
    "AutoConfig.from_pretrained('vinai/bertweet-base')"
)
content = content.replace(
    "AutoModel.from_pretrained('../../model_hub/bertweet-large', local_files_only=True)",
    "AutoModel.from_pretrained('vinai/bertweet-base')"
)
content = content.replace(
    "AutoConfig.from_pretrained('../../model_hub/bertweet-large')",
    "AutoConfig.from_pretrained('vinai/bertweet-base')"
)
content = content.replace(
    "AutoModel.from_pretrained('../../model_hub/bertweet-large')",
    "AutoModel.from_pretrained('vinai/bertweet-base')"
)

with open("/kaggle/working/ZeroStance/src/utils/modeling.py", "w") as f:
    f.write(content)

print("Fixed! Verifying...")
!grep -n "from_pretrained\|model_hub" /kaggle/working/ZeroStance/src/utils/modeling.py

Fixed! Verifying...
14:        self.config = AutoConfig.from_pretrained('vinai/bertweet-base')
15:        self.roberta = AutoModel.from_pretrained('vinai/bertweet-base')
16:        # self.config = AutoConfig.from_pretrained('vinai/bertweet-large')
17:        # self.roberta = AutoModel.from_pretrained('vinai/bertweet-large')


In [27]:
with open("/kaggle/working/ZeroStance/src/pytorchtools.py", "r") as f:
    content = f.read()

content = content.replace("np.Inf", "np.inf")

with open("/kaggle/working/ZeroStance/src/pytorchtools.py", "w") as f:
    f.write(content)

print("Fixed! Verifying...")
!grep -n "np.inf\|np.Inf" /kaggle/working/ZeroStance/src/pytorchtools.py

Fixed! Verifying...
33:        self.val_loss_min = np.inf


In [29]:
# Fix 1: Reduce max_tok_len in config
with open("/kaggle/working/ZeroStance/config/config-roberta_large.txt", "r") as f:
    content = f.read()
print("Current config:\n", content)

content = content.replace("max_tok_len:250", "max_tok_len:128")

with open("/kaggle/working/ZeroStance/config/config-roberta_large.txt", "w") as f:
    f.write(content)
print("\nFixed config:\n", content)

Current config:
 model_select:RoBERTa
bert_lr:1e-5
fc_lr:1e-5
batch_size:64
total_epochs:4
max_tok_len:250
dropout:0.

Fixed config:
 model_select:RoBERTa
bert_lr:1e-5
fc_lr:1e-5
batch_size:64
total_epochs:4
max_tok_len:128
dropout:0.


In [30]:
# Fix 2: Add CUDA_LAUNCH_BLOCKING to see real error if it persists
import subprocess, sys, os

env = os.environ.copy()
env["CUDA_LAUNCH_BLOCKING"] = "1"

result = subprocess.run(
    [sys.executable, "train_model_v2.py",
     "-c", "/kaggle/working/ZeroStance/config/config-roberta_large.txt",
     "-s", "42",
     "-d", "0.1",
     "-train", "/kaggle/working/ZeroStance/data/",
     "-dev", "/kaggle/working/ZeroStance/data/",
     "-test", "/kaggle/working/ZeroStance/data/",
     "-dataset", "covid19",
     "-leave_one_out", "0",
     "-lr1", "1e-5",
     "-lr2", "1e-4",
     "-step", "1",
     "-es_step", "5"],
    cwd="/kaggle/working/ZeroStance/src",
    capture_output=True,
    text=True,
    env=env
)
print("STDOUT:", result.stdout[-3000:])
print("STDERR:", result.stderr[-3000:])
print("Return code:", result.returncode)

STDOUT: ---------
-------------------------------------------------------------------------------------
test classification_report for step: 284
              precision    recall  f1-score   support

     Against     0.6566    0.7160    0.6850       243
       Favor     0.6700    0.7643    0.7140       263
     Neutral     0.8085    0.6463    0.7183       294

    accuracy                         0.7063       800
   macro avg     0.7117    0.7089    0.7058       800
weighted avg     0.7168    0.7063    0.7068       800

results_weighted: (0.711704803960926, 0.70885548041683, 0.7058026085305015, None)
result_against: [np.float64(0.6566037735849056), np.float64(0.7160493827160493), np.float64(0.6850393700787402)]
result_favor: [np.float64(0.67), np.float64(0.7642585551330798), np.float64(0.7140319715808171)]
result_neutral: [np.float64(0.8085106382978723), np.float64(0.6462585034013606), np.float64(0.718336483931947)]
result_overall: [0.711704803960926, 0.70885548041683, 0.70580260853050

In [31]:
# View final results
import pandas as pd
df = pd.read_csv("/kaggle/working/ZeroStance/src/best_results_test_df.csv", header=None)
df.columns = ['split','dataset','step','seed','dropout',
              'against_p','against_r','against_f1',
              'favor_p','favor_r','favor_f1',
              'none_p','none_r','none_f1',
              'overall_p','overall_r','overall_f1']
print(df[['dataset','against_f1','favor_f1','none_f1','overall_f1']].to_string())

   dataset  against_f1  favor_f1   none_f1  overall_f1
0  covid19    0.685039  0.714032  0.718336    0.705803


In [34]:
import shutil, os
os.makedirs("/kaggle/working/saved_model", exist_ok=True)
shutil.copy(
    "/kaggle/working/ZeroStance/src/RoBERTa_seed42.pt",
    "/kaggle/working/saved_model/RoBERTa_covid19_seed42.pt"
)
print("Model saved!")

Model saved!


In [36]:
!pip install wordninja tweet-preprocessor -q

import torch
import torch.nn as nn
import pandas as pd
import re
import wordninja
import preprocessor as p
from transformers import AutoTokenizer, AutoModel, AutoConfig

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 128
LABEL_MAP  = {0: "AGAINST", 1: "FAVOR", 2: "NONE"}

def data_clean(strings, norm_dict={}):
    p.set_options(p.OPT.URL, p.OPT.EMOJI, p.OPT.RESERVED)
    clean_data = p.clean(strings)
    clean_data = re.sub(r"#SemST", "", clean_data)
    clean_data = re.findall(r"[A-Za-z#@]+|[,.!?&/\<>=$]|[0-9]+", clean_data)
    clean_data = [[x.lower()] for x in clean_data]
    for i in range(len(clean_data)):
        if clean_data[i][0] in norm_dict:
            clean_data[i] = norm_dict[clean_data[i][0]].split()
            continue
        if clean_data[i][0].startswith("#") or clean_data[i][0].startswith("@"):
            clean_data[i] = wordninja.split(clean_data[i][0])
    return [j for i in clean_data for j in i]

def prepare_input(topic, text, norm_dict={}, add_prefix=True):
    clean_text   = data_clean(text, norm_dict)
    topic_str    = ('I am in favor of ' + topic + ' ! ') if add_prefix else topic
    clean_target = data_clean(topic_str, norm_dict)
    return ' '.join(clean_text), ' '.join(clean_target)

class roberta_large_classifier(nn.Module):
    def __init__(self, num_labels=3, dropout=0.1):
        super().__init__()
        self.config  = AutoConfig.from_pretrained('vinai/bertweet-base')
        self.roberta = AutoModel.from_pretrained('vinai/bertweet-base')
        self.roberta.pooler = None
        self.dropout = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
        self.linear  = nn.Linear(self.config.hidden_size * 2, self.config.hidden_size)
        self.out     = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        last_hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        eos_token_ind = input_ids.eq(self.config.eos_token_id).nonzero()
        assert len(eos_token_ind) == 3 * len(input_ids)
        b_eos = [eos_token_ind[i][1] for i in range(len(eos_token_ind)) if i % 3 == 0]
        e_eos = [eos_token_ind[i][1] for i in range(len(eos_token_ind)) if (i + 1) % 3 == 0]
        x_atten_clone = attention_mask.clone().detach()
        for begin, end, att, att2 in zip(b_eos, e_eos, attention_mask, x_atten_clone):
            att[begin:]  = 0;  att2[:begin+2] = 0
            att[0]       = 0;  att2[end]      = 0
        txt_l     = attention_mask.sum(1).to(attention_mask.device)
        topic_l   = x_atten_clone.sum(1).to(attention_mask.device)
        txt_vec   = attention_mask.type(torch.FloatTensor).to(attention_mask.device)
        topic_vec = x_atten_clone.type(torch.FloatTensor).to(attention_mask.device)
        txt_mean   = torch.einsum('blh,bl->bh', last_hidden[0], txt_vec)   / txt_l.unsqueeze(1)
        topic_mean = torch.einsum('blh,bl->bh', last_hidden[0], topic_vec) / topic_l.unsqueeze(1)
        cat    = torch.cat((txt_mean, topic_mean), dim=1)
        query  = self.dropout(cat)
        linear = self.relu(self.linear(query))
        return self.out(linear)

def load_model(model_path):
    print("[INFO] Loading model...")
    tokenizer = AutoTokenizer.from_pretrained('vinai/bertweet-base')
    model = roberta_large_classifier(num_labels=3, dropout=0.1)
    state = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    model.eval()
    print(f"✓ Model ready on {DEVICE}")
    return tokenizer, model

def predict_single(tokenizer, model, topic, text, norm_dict={}, add_prefix=True):
    clean_text, clean_target = prepare_input(topic, text, norm_dict, add_prefix)
    encoding = tokenizer(clean_text, clean_target,
                         add_special_tokens=True, max_length=MAX_LENGTH,
                         padding='max_length', truncation=True, return_tensors="pt")
    input_ids      = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
    probs    = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    pred_idx = int(torch.argmax(logits, dim=-1).item())
    result   = {"topic": topic, "text": text, "prediction": LABEL_MAP[pred_idx]}
    for idx, label in LABEL_MAP.items():
        result[f"prob_{label}"] = round(probs[idx], 4)
    return result

def predict_batch(tokenizer, model, df,
                  topic_col="Target 1", text_col="Tweet",
                  norm_dict={}, add_prefix=True, batch_size=32):
    all_preds = []
    all_probs = {label: [] for label in LABEL_MAP.values()}
    rows  = list(df[[topic_col, text_col]].itertuples(index=False))
    total = len(rows)
    for start in range(0, total, batch_size):
        batch         = rows[start: start + batch_size]
        clean_pairs   = [prepare_input(r[0], r[1], norm_dict, add_prefix) for r in batch]
        clean_texts   = [p[0] for p in clean_pairs]
        clean_targets = [p[1] for p in clean_pairs]
        encoding = tokenizer(clean_texts, clean_targets,
                             add_special_tokens=True, max_length=MAX_LENGTH,
                             padding='max_length', truncation=True, return_tensors="pt")
        input_ids      = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs    = torch.softmax(logits, dim=-1).cpu().tolist()
        pred_ids = torch.argmax(logits, dim=-1).cpu().tolist()
        for pred_idx, prob_row in zip(pred_ids, probs):
            all_preds.append(LABEL_MAP[pred_idx])
            for idx, label in LABEL_MAP.items():
                all_probs[label].append(round(prob_row[idx], 4))
        print(f"  Processed {min(start+batch_size, total)}/{total}", end="\r")
    print()
    out = df.copy()
    out["prediction"] = all_preds
    for label in LABEL_MAP.values():
        out[f"prob_{label}"] = all_probs[label]
    return out

In [37]:
tokenizer, model = load_model("/kaggle/working/ZeroStance/src/RoBERTa_seed42.pt")

[INFO] Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.bias                    | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model ready on cuda


In [38]:
# COVID-19 relevant topics
r1 = predict_single(tokenizer, model, "face masks",
                    "Everyone should wear masks to protect others from COVID.")
r2 = predict_single(tokenizer, model, "face masks",
                    "Masks are useless and just a way to control people.")
r3 = predict_single(tokenizer, model, "stay at home orders",
                    "I went to the beach today, lockdowns are stupid.")

for r in [r1, r2, r3]:
    print(f"[{r['prediction']}] AGAINST={r['prob_AGAINST']} FAVOR={r['prob_FAVOR']} NONE={r['prob_NONE']}")
    print(f"  Topic: {r['topic']}")
    print(f"  Text : {r['text']}\n")

[FAVOR] AGAINST=0.0091 FAVOR=0.9754 NONE=0.0155
  Topic: face masks
  Text : Everyone should wear masks to protect others from COVID.

[AGAINST] AGAINST=0.978 FAVOR=0.0103 NONE=0.0117
  Topic: face masks
  Text : Masks are useless and just a way to control people.

[AGAINST] AGAINST=0.7341 FAVOR=0.1647 NONE=0.1013
  Topic: stay at home orders
  Text : I went to the beach today, lockdowns are stupid.



In [39]:
df = pd.read_csv("/kaggle/working/ZeroStance/data/covid19/raw_test_all_onecol.csv", 
                 encoding='ISO-8859-1')
out_df = predict_batch(tokenizer, model, df, topic_col="Target 1", text_col="Tweet")
out_df.to_csv("/kaggle/working/predictions_covid19.csv", index=False)
out_df[["Target 1", "Tweet", "Stance 1", "prediction"]].head(10)

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


  Processed 800/800


,Target 1,Tweet,Stance 1,prediction
0,face masks,Maybe if people want to walk around without ma...,FAVOR,FAVOR
1,face masks,@Constance8News Deepest condolences on your lo...,FAVOR,FAVOR
2,face masks,We live in an appartment block and #facemasks ...,FAVOR,FAVOR
3,face masks,@TimWilsonMP Yes perhaps these Covidiots youâ...,FAVOR,FAVOR
4,face masks,@josephcollins77 @davidmweissman @WhirledCitiz...,FAVOR,NONE
5,face masks,Anyone have any tips for someone with anxiety/...,FAVOR,FAVOR
6,face masks,@RuekaA @cher I know the literature says child...,FAVOR,FAVOR
7,face masks,"@hoaghealth Your own doctor, Dr. Jeff Barke, i...",FAVOR,FAVOR
8,face masks,Went to IKEA today in Cardiff. Superb job by a...,FAVOR,FAVOR
9,face masks,@DrEricDing I flew a few days ago. It was made...,FAVOR,FAVOR


In [40]:
import zipfile, os

files_to_zip = [
    "/kaggle/working/ZeroStance/src/RoBERTa_seed42.pt",
    "/kaggle/working/ZeroStance/src/train_model_v2.py",
    "/kaggle/working/ZeroStance/src/pytorchtools.py",
    "/kaggle/working/ZeroStance/src/utils/modeling.py",
    "/kaggle/working/ZeroStance/src/utils/data_helper.py",
    "/kaggle/working/ZeroStance/src/utils/preprocessing.py",
    "/kaggle/working/ZeroStance/config/config-roberta_large.txt",
    "/kaggle/working/ZeroStance/src/best_results_test_df.csv",
    "/kaggle/working/ZeroStance/src/results_test_df.csv",
]

with zipfile.ZipFile("/kaggle/working/stance_detection_project.zip", "w") as zf:
    for f in files_to_zip:
        if os.path.exists(f):
            zf.write(f, os.path.basename(f))
            print(f"Added: {f}")
        else:
            print(f"NOT FOUND: {f}")

print("\nZip created! Download: stance_detection_project.zip")

Added: /kaggle/working/ZeroStance/src/RoBERTa_seed42.pt
Added: /kaggle/working/ZeroStance/src/train_model_v2.py
Added: /kaggle/working/ZeroStance/src/pytorchtools.py
Added: /kaggle/working/ZeroStance/src/utils/modeling.py
Added: /kaggle/working/ZeroStance/src/utils/data_helper.py
Added: /kaggle/working/ZeroStance/src/utils/preprocessing.py
Added: /kaggle/working/ZeroStance/config/config-roberta_large.txt
Added: /kaggle/working/ZeroStance/src/best_results_test_df.csv
Added: /kaggle/working/ZeroStance/src/results_test_df.csv

Zip created! Download: stance_detection_project.zip
